# Treino no Kaggle (GPU T4)

Clona o repositório, instala as dependências, prepara os dados, treina os baselines e o modelo, avalia e empacota os resultados.

**Antes de rodar:** em *Settings*, ligue **Accelerator: GPU T4** e **Internet**. Para publicar o modelo, crie o segredo `HF_TOKEN` em *Add-ons > Secrets*.

## 1. Clonar o repositório

Ajuste `REPO_URL` se o repositório estiver em outro endereço.

In [ ]:
REPO_URL = "https://github.com/ItaloSLeao/ml-image-classifier.git"

%cd /kaggle/working
!rm -rf br-image-classifier
!git clone --depth 1 {REPO_URL} br-image-classifier
%cd br-image-classifier
!git log --oneline -1

## 2. Dependências

Instala só `transformers` e `huggingface_hub` nas versões de `requirements.txt`. O PyTorch com CUDA da imagem do Kaggle é mantido.

In [ ]:
!pip install -q $(grep -E '^(transformers|huggingface_hub)==' requirements.txt)

import torch, torchvision, transformers
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| transformers", transformers.__version__)
assert torch.cuda.is_available(), "GPU desligada: ligue o acelerador T4 em Settings"
print(torch.cuda.get_device_name(0))

## 3. Testes rápidos

In [ ]:
!python -m pytest -q tests

## 4. Dados

Baixa o dataset da Mendeley (cerca de 80 MB, alguns minutos), gera os recortes e confere o split versionado em `data/splits.json`.

In [ ]:
!python -m src.data

## 5. Baseline 1: classe majoritária

In [ ]:
!python -m src.evaluate --majority

## 6. Baseline 2: backbone congelado + cabeça linear

In [ ]:
!python -m src.train --name convnext-tiny-linear-probe --linear-probe
!python -m src.evaluate --name convnext-tiny-linear-probe

## 7. Modelo: fine-tuning completo

In [ ]:
!python -m src.train --name convnext-tiny-finetune
!python -m src.evaluate --name convnext-tiny-finetune

## 8. Tabelas e gráficos

Atualiza `results/comparacao.md`, o README e o model card a partir dos JSON.

In [ ]:
!python -m src.report > /dev/null

from IPython.display import Image, Markdown, display
display(Markdown(open("results/comparacao.md", encoding="utf-8").read()))
display(Image("results/convnext-tiny-finetune_confusao.png"))
display(Image("results/convnext-tiny-finetune_erros.png"))

## 9. Inferência de exemplo

In [ ]:
import json
from src import data
from src.predict import predict

item = data.load_splits()["splits"]["test"][0]
out = predict(data.CROPS_DIR / item["path"], model="checkpoints/convnext-tiny-finetune")
print("real:", item["label"])
print(json.dumps(out, ensure_ascii=False, indent=1))

## 10. Empacotar os resultados

Baixe `results.zip` na aba *Output*, extraia na raiz do repositório local, rode `python -m src.report` e faça o commit.

In [ ]:
!zip -qr /kaggle/working/results.zip results && ls -lh /kaggle/working/results.zip

## 11. Publicar no Hugging Face Hub (opcional)

Só rode depois de conferir os resultados. O token é lido do Kaggle Secrets e não é impresso.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!python -m src.push --name convnext-tiny-finetune